# Simple Concurrent Workflow with Microsoft Agent Framework

This tutorial demonstrates how to build a **fan-out/fan-in concurrent workflow** using the [Microsoft Agent Framework](https://learn.microsoft.com/en-us/agent-framework/).

## Overview

This workflow implements a common pattern where:
1. **Dispatcher** receives input and distributes it to multiple parallel executors
2. **Min** and **Max** executors run concurrently to calculate the minimum and maximum values
3. **Aggregator** collects results from all parallel executors and yields the final output

### Use Cases
- Parallel data processing and aggregation
- Concurrent statistical calculations
- Splitting and merging workflow branches
- Any scenario requiring multiple independent computations on the same data

**Reference**: This example is based on the [official Microsoft tutorial](https://learn.microsoft.com/en-us/agent-framework/tutorials/workflows/simple-concurrent-workflow?pivots=programming-language-python)

## Prerequisites

Before running this notebook, ensure you have the Microsoft Agent Framework installed:

```bash
pip install agent_framework
```

### Key Concepts

- **Executor**: A class that processes input and produces output, decorated with handlers
- **WorkflowBuilder**: Constructs the workflow by defining connections between executors
- **Fan-out**: One executor sends data to multiple parallel executors
- **Fan-in**: Multiple executors send results to a single aggregator executor
- **WorkflowContext**: Provides methods to send messages and yield outputs within the workflow

## Step 1: Import Dependencies

Import the necessary modules from the Microsoft Agent Framework and Python standard library.

- `Executor`: Base class for creating workflow nodes
- `WorkflowBuilder`: API for constructing workflow graphs
- `WorkflowContext`: Context for message passing and output yielding
- `WorkflowOutputEvent`: Event type for capturing workflow results
- `WorkflowViz`: Utility for visualizing the workflow structure

In [15]:
import asyncio
import random

from agent_framework import Executor, WorkflowBuilder, WorkflowContext, WorkflowOutputEvent, handler
from typing_extensions import Never
from agent_framework import WorkflowBuilder, WorkflowViz

## Step 2: Create the Dispatcher Executor

The **Dispatcher** is the entry point of our workflow. Its sole purpose is to:
1. Validate the input (must be a non-empty list of integers)
2. Dispatch the input to downstream executors

### Key Features
- Uses the `@handler` decorator to define the message processing method
- Validates input and raises `RuntimeError` for invalid data
- Uses `ctx.send_message()` to pass data to connected executors

In a production scenario, the dispatcher could:
- Parse and validate complex inputs
- Transform data before distribution
- Route to different executors based on input characteristics

In [16]:
class Dispatcher(Executor):
    """
    The sole purpose of this executor is to dispatch the input of the workflow to
    other executors.
    """

    @handler
    async def handle(self, numbers: list[int], ctx: WorkflowContext[list[int]]):
        if not numbers:
            raise RuntimeError("Input must be a valid list of integers.")

        await ctx.send_message(numbers)

## Step 3: Create Parallel Processing Executors

These executors run **concurrently** to perform different calculations on the same input data.

### Min Executor
- Calculates the minimum value from the list of integers
- Returns a `float` type (for flexibility in handling edge cases)

### Max Executor
- Calculates the maximum value from the list of integers
- Returns an `int` type

### Design Pattern
Both executors follow the same pattern:
1. Receive the list of integers from the Dispatcher
2. Perform their specific calculation
3. Send the result via `ctx.send_message()`

The framework automatically routes the output to the Aggregator executor.

In [17]:
class Min(Executor):
    """Calculate the Minum of a list of integers."""

    @handler
    async def handle(self, numbers: list[int], ctx: WorkflowContext[float]):
        minimum: float = min(numbers)
        await ctx.send_message(minimum)


class Max(Executor):
    """Calculate the Max of a list of integers."""

    @handler
    async def handle(self, numbers: list[int], ctx: WorkflowContext[int]):
        maximum: int = max(numbers)
        await ctx.send_message(maximum)

## Step 4: Create the Aggregator Executor

The **Aggregator** is the final executor in our workflow. It collects results from all parallel executors and produces the final output.

### Key Features
- **Fan-in Pattern**: Automatically receives messages from upstream executors as a list
- **Type Safety**: The type annotation `list[int | float]` tells the framework what types to expect
- **Output Yielding**: Uses `ctx.yield_output()` to emit the final workflow result

### How Fan-in Works
When multiple executors (Min and Max) send messages to the Aggregator:
1. The framework collects all messages
2. Automatically delivers them as a list to the Aggregator's handler
3. The order in the list depends on execution timing (non-deterministic)

### Never Type
`WorkflowContext[Never, list[int | float]]` indicates:
- First `Never`: This executor doesn't send messages to other executors
- Second type: The output type that will be yielded

In [18]:
class Aggregator(Executor):
    """Aggregate the results from the different tasks and yield the final output."""

    @handler
    async def handle(self, results: list[int | float], ctx: WorkflowContext[Never, list[int | float]]):
        """Receive the results from the source executors.

        The framework will automatically collect messages from the source executors
        and deliver them as a list.

        Args:
            results (list[int | float]): execution results from upstream executors.
                The type annotation must be a list of union types that the upstream
                executors will produce.
            ctx (WorkflowContext[Never, list[int | float]]): A workflow context that can yield the final output.
        """
        await ctx.yield_output(results)

## Step 5: Build and Execute the Workflow

Now we'll wire everything together using the `WorkflowBuilder` and run the workflow.

### Workflow Construction Steps

1. **Create Executor Instances**: Instantiate all executors with unique IDs
2. **Set Start Executor**: Define the entry point (Dispatcher)
3. **Add Fan-out Edges**: Connect Dispatcher to both Min and Max executors
4. **Add Fan-in Edges**: Connect Min and Max to the Aggregator
5. **Build the Workflow**: Compile the workflow graph

### Execution Flow

```
Input List ──► Dispatcher ──┬──► Min ──┐
                            │          ├──► Aggregator ──► Output
                            └──► Max ──┘
```

### Running the Workflow
- Use `workflow.run_stream()` for asynchronous execution
- Listen for `WorkflowOutputEvent` to capture results
- Generate a visualization using `WorkflowViz`

In [19]:
async def main() -> None:
    # 1) Create the executors
    dispatcher = Dispatcher(id="dispatcher")
    minimum = Min(id="min")
    maximum = Max(id="max")
    aggregator = Aggregator(id="aggregator")

    # 2) Build a simple fan out and fan in workflow
    workflow = (
        WorkflowBuilder()
        .set_start_executor(dispatcher)
        .add_fan_out_edges(dispatcher, [minimum, maximum])
        .add_fan_in_edges([minimum, maximum], aggregator)
        .build()
    )
    # 3) Run the workflow
    output: list[int | float] | None = None
    async for event in workflow.run_stream([random.randint(1, 100) for _ in range(10)]):
        if isinstance(event, WorkflowOutputEvent):
            output = event.data

    if output is not None:
        print(output)
    viz = WorkflowViz(workflow)
    print(viz.save_png("simple_concurrent_workflow.png"))

## Step 6: Run the Workflow

Execute the workflow with a randomly generated list of 10 integers (values between 1-100).

### Expected Output
The workflow will output a list containing two elements:
- `[minimum_value, maximum_value]` (order may vary)

A workflow visualization will also be saved as `simple_concurrent_workflow.png`.

In [20]:
await main()

[13, 93]
simple_concurrent_workflow.png


## Summary

Congratulations! You've successfully built a **fan-out/fan-in concurrent workflow** using the Microsoft Agent Framework.

### What We Learned

1. **Concurrent Execution**: The Min and Max executors run in parallel, improving efficiency
2. **Fan-out Pattern**: Dispatcher distributes work to multiple executors simultaneously
3. **Fan-in Pattern**: Aggregator collects results from multiple sources automatically
4. **Type Safety**: Proper type annotations ensure data flows correctly through the workflow
5. **Visualization**: The framework provides tools to visualize workflow structure

### Next Steps

- Extend this pattern with more parallel executors (e.g., Mean, Median, Mode)
- Add error handling and retry logic to executors
- Explore conditional routing in workflows
- Build more complex multi-stage workflows

### Resources

- [Microsoft Agent Framework Documentation](https://learn.microsoft.com/en-us/agent-framework/)
- [Simple Concurrent Workflow Tutorial](https://learn.microsoft.com/en-us/agent-framework/tutorials/workflows/simple-concurrent-workflow?pivots=programming-language-python)

---

**Share this tutorial on**: [LinkedIn](#) | [GitHub](#) | [Twitter](#)

*Built with ❤️ using Microsoft Agent Framework*